# 目标检测工程：从框坐标到 IoU、NMS、AP/mAP

目标检测最容易出现“模型没变、指标却变了”的问题：`xywh` 被当成 `xyxy`、resize padding 没有逆映射、NMS 跨类别抑制、相同真值被重复匹配、ignore/crowd 被当普通负例、AP 插值协议不一致。

本 notebook 不调用检测框架的黑盒评估器，而是在受控样例上从零实现坐标变换、IoU、class-aware greedy NMS、Soft-NMS 教学版、one-to-one 匹配、PR/AP/mAP 和版本化后处理，并用反例断言边界。

## 1. 检测系统的外部合同

```text
原图尺寸 / 方向
  -> resize + pad（保存 scale 与 pad）
  -> 模型输出 boxes/logits
  -> 解码到模型输入坐标
  -> clip / 去退化框 / 置信阈值
  -> 逆映射到原图
  -> class-aware NMS 或 Soft-NMS
  -> 固定协议的 one-to-one matching
  -> PR、AP、mAP 与延迟报告
```

每个阶段都应写入 trace：坐标约定、模型输入尺寸、阈值、NMS 类型、IoU 阈值和代码版本。最终框没有这些元数据，就很难解释线上与离线差异。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
from collections import Counter, defaultdict  # 导入本单元所需的依赖。
import time  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 坐标统一为原图像素、左上角原点、半开区间 [x1,y1,x2,y2)。
GROUND_TRUTH = [  # 计算并保存当前步骤的中间状态。
    {"image_id": "img-1", "class_id": 0, "box": [10, 10, 50, 50], "ignore": False, "crowd": False},  # 执行当前语句以推进本节示例。
    {"image_id": "img-1", "class_id": 1, "box": [60, 10, 95, 45], "ignore": False, "crowd": False},  # 执行当前语句以推进本节示例。
    {"image_id": "img-1", "class_id": 0, "box": [0, 0, 8, 8], "ignore": True, "crowd": False},  # 执行当前语句以推进本节示例。
    {"image_id": "img-2", "class_id": 0, "box": [20, 20, 60, 70], "ignore": False, "crowd": False},  # 执行当前语句以推进本节示例。
    {"image_id": "img-2", "class_id": 1, "box": [65, 25, 95, 65], "ignore": False, "crowd": True},  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
PREDICTIONS = [  # 计算并保存当前步骤的中间状态。
    {"image_id": "img-1", "class_id": 0, "score": .95, "box": [11, 11, 49, 50]},  # 执行当前语句以推进本节示例。
    {"image_id": "img-1", "class_id": 0, "score": .80, "box": [12, 12, 48, 49]},  # duplicate；中文说明：该行遵循既定约束。
    {"image_id": "img-1", "class_id": 1, "score": .90, "box": [62, 12, 94, 44]},  # 执行当前语句以推进本节示例。
    {"image_id": "img-1", "class_id": 1, "score": .30, "box": [0, 70, 20, 90]},   # background；中文说明：该行遵循既定约束。
    {"image_id": "img-1", "class_id": 0, "score": .40, "box": [0, 0, 7, 7]},      # ignore overlap；中文说明：该行遵循既定约束。
    {"image_id": "img-2", "class_id": 0, "score": .88, "box": [19, 22, 61, 69]},  # 执行当前语句以推进本节示例。
    {"image_id": "img-2", "class_id": 0, "score": .65, "box": [65, 25, 95, 65]}, # wrong class；中文说明：该行遵循既定约束。
    {"image_id": "img-2", "class_id": 1, "score": .72, "box": [66, 26, 94, 64]}, # crowd；中文说明：该行遵循既定约束。
    {"image_id": "img-2", "class_id": 1, "score": .60, "box": [67, 27, 93, 63]}, # crowd again；中文说明：该行遵循既定约束。
]  # 执行当前语句以推进本节示例。
IMAGE_SHAPES = {"img-1": (100, 100), "img-2": (100, 100)}  # (height,width)；中文说明：该行遵循既定约束。
print("GT:", len(GROUND_TRUTH), "predictions:", len(PREDICTIONS))  # 执行当前语句以推进本节示例。


## 2. 框坐标、clip 和 resize 逆映射

本例选择浮点 `xyxy=[x1,y1,x2,y2]` 半开区间，因此宽高为 `x2-x1,y2-y1`；另一套库可能使用闭区间并出现 `+1`。协议不一致会系统性改变小目标 IoU。

letterbox resize 不能只保存缩放后图片：必须保存 `scale、pad_x、pad_y、原图尺寸、目标尺寸`。逆映射先减 padding 再除 scale，最后 clip 到原图。退化框（宽或高为 0）保留为可诊断输入，但面积和 IoU 都是 0；负宽高则直接拒绝。

In [ ]:
def as_boxes(boxes):  # 定义本节可复用的核心函数。
    array = np.asarray(boxes, dtype=np.float64)  # 计算并保存当前步骤的中间状态。
    if array.size == 0:  # 按当前条件选择后续控制路径。
        return np.empty((0, 4), dtype=np.float64)  # 返回当前分支计算出的结果。
    if array.ndim == 1:  # 按当前条件选择后续控制路径。
        if array.shape != (4,):  # 按当前条件选择后续控制路径。
            raise ValueError("单个 box 必须有四个坐标")  # 遇到非法合同立即显式失败。
        array = array.reshape(1, 4)  # 计算并保存当前步骤的中间状态。
    if array.ndim != 2 or array.shape[1] != 4 or not np.isfinite(array).all():  # 按当前条件选择后续控制路径。
        raise ValueError("boxes 必须是有限的 [N,4]")  # 遇到非法合同立即显式失败。
    if np.any(array[:, 2] < array[:, 0]) or np.any(array[:, 3] < array[:, 1]):  # 按当前条件选择后续控制路径。
        raise ValueError("xyxy 不能出现负宽或负高")  # 遇到非法合同立即显式失败。
    return array  # 返回当前分支计算出的结果。

def validate_image_hw(image_hw, name="image_hw"):  # 定义本节可复用的核心函数。
    array = np.asarray(image_hw, dtype=float)  # 计算并保存当前步骤的中间状态。
    if array.shape != (2,) or not np.isfinite(array).all() or np.any(array <= 0):  # 按当前条件选择后续控制路径。
        raise ValueError(f"{name} 必须是两个有限正数 (height,width)")  # 遇到非法合同立即显式失败。
    return float(array[0]), float(array[1])  # 返回当前分支计算出的结果。

def validate_probability_threshold(value, name):  # 定义本节可复用的核心函数。
    if not np.isfinite(value) or not 0 <= value <= 1:  # 按当前条件选择后续控制路径。
        raise ValueError(f"{name} 必须是 [0,1] 有限数")  # 遇到非法合同立即显式失败。

def validate_detection_arrays(boxes, scores, class_ids):  # 定义本节可复用的核心函数。
    boxes = as_boxes(boxes)  # 计算并保存当前步骤的中间状态。
    scores = np.asarray(scores, dtype=float)  # 计算并保存当前步骤的中间状态。
    raw_classes = np.asarray(class_ids)  # 计算并保存当前步骤的中间状态。
    if scores.ndim != 1 or raw_classes.ndim != 1:  # 按当前条件选择后续控制路径。
        raise ValueError("scores/class_ids 必须是一维")  # 遇到非法合同立即显式失败。
    if len(boxes) != len(scores) or len(boxes) != len(raw_classes):  # 按当前条件选择后续控制路径。
        raise ValueError("boxes/scores/class_ids 长度不一致")  # 遇到非法合同立即显式失败。
    if not np.isfinite(scores).all() or np.any((scores < 0) | (scores > 1)):  # 按当前条件选择后续控制路径。
        raise ValueError("scores 必须是 [0,1] 有限概率")  # 遇到非法合同立即显式失败。
    try:  # 尝试执行可能失败的受控操作。
        numeric_classes = raw_classes.astype(float)  # 计算并保存当前步骤的中间状态。
    except (TypeError, ValueError):  # 捕获预期异常并验证失败分支。
        raise ValueError("class_ids 必须是有限整数") from None  # 遇到非法合同立即显式失败。
    if not np.isfinite(numeric_classes).all() or not np.all(numeric_classes == np.floor(numeric_classes)):  # 按当前条件选择后续控制路径。
        raise ValueError("class_ids 必须是有限整数")  # 遇到非法合同立即显式失败。
    return boxes, scores, numeric_classes.astype(np.int64)  # 返回当前分支计算出的结果。

def xywh_to_xyxy(boxes):  # 定义本节可复用的核心函数。
    array = np.asarray(boxes, dtype=np.float64)  # 计算并保存当前步骤的中间状态。
    if array.size == 0:  # 按当前条件选择后续控制路径。
        return np.empty((0, 4), dtype=np.float64)  # 返回当前分支计算出的结果。
    if array.ndim == 1:  # 按当前条件选择后续控制路径。
        if array.shape != (4,):  # 按当前条件选择后续控制路径。
            raise ValueError("单个 xywh 必须有四个值")  # 遇到非法合同立即显式失败。
        array = array.reshape(1, 4)  # 计算并保存当前步骤的中间状态。
    if array.ndim != 2 or array.shape[1] != 4 or not np.isfinite(array).all():  # 按当前条件选择后续控制路径。
        raise ValueError("xywh 必须是有限 [N,4]")  # 遇到非法合同立即显式失败。
    if np.any(array[:, 2:] < 0):  # 按当前条件选择后续控制路径。
        raise ValueError("xywh 宽高必须非负")  # 遇到非法合同立即显式失败。
    out = array.copy()  # 计算并保存当前步骤的中间状态。
    out[:, 2] = array[:, 0] + array[:, 2]  # 计算并保存当前步骤的中间状态。
    out[:, 3] = array[:, 1] + array[:, 3]  # 计算并保存当前步骤的中间状态。
    return out  # 返回当前分支计算出的结果。

def xyxy_to_xywh(boxes):  # 定义本节可复用的核心函数。
    boxes = as_boxes(boxes)  # 计算并保存当前步骤的中间状态。
    out = boxes.copy()  # 计算并保存当前步骤的中间状态。
    out[:, 2] = boxes[:, 2] - boxes[:, 0]  # 计算并保存当前步骤的中间状态。
    out[:, 3] = boxes[:, 3] - boxes[:, 1]  # 计算并保存当前步骤的中间状态。
    return out  # 返回当前分支计算出的结果。

def clip_boxes(boxes, image_hw):  # 定义本节可复用的核心函数。
    boxes = as_boxes(boxes).copy()  # 计算并保存当前步骤的中间状态。
    height, width = validate_image_hw(image_hw)  # 计算并保存当前步骤的中间状态。
    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, width)  # 计算并保存当前步骤的中间状态。
    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, height)  # 计算并保存当前步骤的中间状态。
    return boxes  # 返回当前分支计算出的结果。

def letterbox_map(boxes, source_hw, target_hw):  # 定义本节可复用的核心函数。
    source_h, source_w = validate_image_hw(source_hw, "source_hw")  # 计算并保存当前步骤的中间状态。
    target_h, target_w = validate_image_hw(target_hw, "target_hw")  # 计算并保存当前步骤的中间状态。
    scale = min(target_w / source_w, target_h / source_h)  # 计算并保存当前步骤的中间状态。
    pad_x = (target_w - source_w * scale) / 2.0  # 计算并保存当前步骤的中间状态。
    pad_y = (target_h - source_h * scale) / 2.0  # 计算并保存当前步骤的中间状态。
    mapped = as_boxes(boxes).copy() * scale  # 计算并保存当前步骤的中间状态。
    mapped[:, [0, 2]] += pad_x  # 计算并保存当前步骤的中间状态。
    mapped[:, [1, 3]] += pad_y  # 计算并保存当前步骤的中间状态。
    meta = {"scale": scale, "pad_x": pad_x, "pad_y": pad_y,  # 计算并保存当前步骤的中间状态。
            "source_hw": (source_h, source_w), "target_hw": (target_h, target_w)}  # 执行当前语句以推进本节示例。
    return mapped, meta  # 返回当前分支计算出的结果。

def letterbox_inverse(boxes, meta):  # 定义本节可复用的核心函数。
    required = {"scale", "pad_x", "pad_y", "source_hw"}  # 计算并保存当前步骤的中间状态。
    if not required.issubset(meta) or not np.isfinite([meta["scale"], meta["pad_x"], meta["pad_y"]]).all():  # 按当前条件选择后续控制路径。
        raise ValueError("letterbox meta 缺失或含非有限值")  # 遇到非法合同立即显式失败。
    if meta["scale"] <= 0:  # 按当前条件选择后续控制路径。
        raise ValueError("letterbox scale 必须为正")  # 遇到非法合同立即显式失败。
    restored = as_boxes(boxes).copy()  # 计算并保存当前步骤的中间状态。
    restored[:, [0, 2]] = (restored[:, [0, 2]] - meta["pad_x"]) / meta["scale"]  # 计算并保存当前步骤的中间状态。
    restored[:, [1, 3]] = (restored[:, [1, 3]] - meta["pad_y"]) / meta["scale"]  # 计算并保存当前步骤的中间状态。
    return clip_boxes(restored, meta["source_hw"])  # 返回当前分支计算出的结果。

probe = np.array([[10, 5, 90, 45]], dtype=float)  # 计算并保存当前步骤的中间状态。
mapped, resize_meta = letterbox_map(probe, (50, 100), (128, 128))  # 计算并保存当前步骤的中间状态。
restored = letterbox_inverse(mapped, resize_meta)  # 计算并保存当前步骤的中间状态。
assert np.allclose(restored, probe)  # 用受控断言验证关键不变量。
assert np.allclose(xywh_to_xyxy(xyxy_to_xywh(probe)), probe)  # 用受控断言验证关键不变量。
print("mapped:", mapped.round(2).tolist(), "meta:", resize_meta)  # 执行当前语句以推进本节示例。


## 3. IoU：几何合同的第一道单测

$IoU(A,B)=|A\cap B|/|A\cup B|$。实现应向量化，并明确空并集、接触边界、退化框的返回值。本例把它们定义为 0。对旋转框、mask、3D box 则需要另一套几何与协议，不能复用轴对齐框函数冒充。

In [ ]:
def box_area(boxes):  # 定义本节可复用的核心函数。
    boxes = as_boxes(boxes)  # 计算并保存当前步骤的中间状态。
    return np.maximum(0.0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0.0, boxes[:, 3] - boxes[:, 1])  # 返回当前分支计算出的结果。

def box_iou(boxes_a, boxes_b):  # 定义本节可复用的核心函数。
    a, b = as_boxes(boxes_a), as_boxes(boxes_b)  # 计算并保存当前步骤的中间状态。
    top_left = np.maximum(a[:, None, :2], b[None, :, :2])  # 计算并保存当前步骤的中间状态。
    bottom_right = np.minimum(a[:, None, 2:], b[None, :, 2:])  # 计算并保存当前步骤的中间状态。
    wh = np.maximum(0.0, bottom_right - top_left)  # 计算并保存当前步骤的中间状态。
    intersection = wh[..., 0] * wh[..., 1]  # 计算并保存当前步骤的中间状态。
    union = box_area(a)[:, None] + box_area(b)[None, :] - intersection  # 计算并保存当前步骤的中间状态。
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)  # 返回当前分支计算出的结果。

assert np.isclose(box_iou([[0, 0, 10, 10]], [[0, 0, 10, 10]])[0, 0], 1.0)  # 用受控断言验证关键不变量。
assert box_iou([[0, 0, 10, 10]], [[10, 0, 20, 10]])[0, 0] == 0.0  # 用受控断言验证关键不变量。
assert box_iou([[1, 1, 1, 5]], [[1, 1, 1, 5]])[0, 0] == 0.0  # 用受控断言验证关键不变量。
print(box_iou([[0, 0, 10, 10], [0, 0, 5, 5]], [[0, 0, 10, 10]]).ravel())  # 执行当前语句以推进本节示例。


## 4. Greedy NMS：排序稳定性与类别边界

标准 greedy NMS 按置信度从高到低选择框，再删除与它 IoU 超阈值的候选。相同分数必须有稳定 tie-break，否则不同硬件/并行顺序可能产生不同输出。

默认应该 class-aware：猫框不应抑制同位置的狗框。若业务确实要求 class-agnostic，需要作为显式、版本化配置。NMS 阈值与 score 阈值必须在 validation 上联合调节。

In [ ]:
def greedy_class_aware_nms(boxes, scores, class_ids, iou_threshold=0.5):  # 定义本节可复用的核心函数。
    boxes, scores, class_ids = validate_detection_arrays(boxes, scores, class_ids)  # 计算并保存当前步骤的中间状态。
    validate_probability_threshold(iou_threshold, "iou_threshold")  # 执行当前语句以推进本节示例。
    keep = []  # 计算并保存当前步骤的中间状态。
    for class_id in np.unique(class_ids):  # 遍历输入元素以累积或检查结果。
        indices = np.flatnonzero(class_ids == class_id)  # 计算并保存当前步骤的中间状态。
        order = indices[np.lexsort((indices, -scores[indices]))]  # 计算并保存当前步骤的中间状态。
        while len(order):  # 在终止条件满足前持续推进状态。
            current = int(order[0])  # 计算并保存当前步骤的中间状态。
            keep.append(current)  # 执行当前语句以推进本节示例。
            if len(order) == 1:  # 按当前条件选择后续控制路径。
                break  # 调整当前循环或占位控制流。
            overlaps = box_iou(boxes[[current]], boxes[order[1:]])[0]  # 计算并保存当前步骤的中间状态。
            order = order[1:][overlaps <= iou_threshold]  # 计算并保存当前步骤的中间状态。
    return np.array(sorted(keep, key=lambda i: (-scores[i], i)), dtype=int)  # 返回当前分支计算出的结果。

probe_boxes = np.array([[0, 0, 10, 10], [1, 1, 9, 9], [0, 0, 10, 10]], float)  # 计算并保存当前步骤的中间状态。
probe_scores = np.array([.9, .8, .85])  # 计算并保存当前步骤的中间状态。
probe_classes = np.array([0, 0, 1])  # 计算并保存当前步骤的中间状态。
kept = greedy_class_aware_nms(probe_boxes, probe_scores, probe_classes, .5)  # 计算并保存当前步骤的中间状态。
assert kept.tolist() == [0, 2]  # 用受控断言验证关键不变量。
print("kept indices:", kept.tolist())  # 执行当前语句以推进本节示例。


## 5. Soft-NMS：衰减而不是删除

拥挤场景中，硬删除可能把邻近真实目标一起移除。Soft-NMS 根据 IoU 衰减分数，再重新排序。下面实现线性衰减的教学版本；原论文还讨论 Gaussian 形式。

注意：衰减后 score 的分布改变，旧置信阈值和校准不再有效。生产切换 NMS 算法时，应连同阈值、评估协议和后处理版本一起发布。

In [ ]:
def linear_soft_nms(boxes, scores, class_ids, iou_threshold=0.5, min_score=0.05):  # 定义本节可复用的核心函数。
    boxes, adjusted, class_ids = validate_detection_arrays(boxes, scores, class_ids)  # 计算并保存当前步骤的中间状态。
    adjusted = adjusted.copy()  # 计算并保存当前步骤的中间状态。
    validate_probability_threshold(iou_threshold, "iou_threshold")  # 执行当前语句以推进本节示例。
    validate_probability_threshold(min_score, "min_score")  # 执行当前语句以推进本节示例。
    selected = []  # 计算并保存当前步骤的中间状态。
    for class_id in np.unique(class_ids):  # 遍历输入元素以累积或检查结果。
        remaining = list(np.flatnonzero(class_ids == class_id))  # 计算并保存当前步骤的中间状态。
        while remaining:  # 在终止条件满足前持续推进状态。
            current = min(remaining, key=lambda i: (-adjusted[i], i))  # 计算并保存当前步骤的中间状态。
            remaining.remove(current)  # 执行当前语句以推进本节示例。
            if adjusted[current] < min_score:  # 按当前条件选择后续控制路径。
                break  # 调整当前循环或占位控制流。
            selected.append((int(current), float(adjusted[current])))  # 执行当前语句以推进本节示例。
            if remaining:  # 按当前条件选择后续控制路径。
                overlaps = box_iou(boxes[[current]], boxes[remaining])[0]  # 计算并保存当前步骤的中间状态。
                for index, overlap in zip(remaining, overlaps):  # 遍历输入元素以累积或检查结果。
                    if overlap > iou_threshold:  # 按当前条件选择后续控制路径。
                        adjusted[index] *= 1.0 - overlap  # 计算并保存当前步骤的中间状态。
                remaining = [index for index in remaining if adjusted[index] >= min_score]  # 计算并保存当前步骤的中间状态。
    return sorted(selected, key=lambda item: (-item[1], item[0])), adjusted  # 返回当前分支计算出的结果。

soft_kept, soft_scores = linear_soft_nms(probe_boxes, probe_scores, probe_classes, .5, .01)  # 计算并保存当前步骤的中间状态。
assert soft_scores[1] < probe_scores[1]  # 用受控断言验证关键不变量。
assert soft_scores[2] == probe_scores[2]  # 用受控断言验证关键不变量。
print(soft_kept)  # 执行当前语句以推进本节示例。


## 6. 一对一匹配：高分预测先占用真值

对某个类别和 IoU 阈值，预测按 score 全局降序。每个普通 GT 最多匹配一次：首个达到阈值的预测记 TP，之后覆盖同一 GT 的预测记 duplicate FP。没有普通匹配、但覆盖 ignore/crowd 区域的预测不进入 TP/FP。

下面对 crowd 使用“可吸收多个预测”的简化规则，用来展示边界；它**不是 COCO 官方评估器的完整复刻**，没有 area range、maxDets、mask crowd IoU 等细节。正式 COCO 报告应使用固定版本 `pycocotools` 并记录参数。

In [ ]:
def crowd_overlap(det_boxes, crowd_boxes):  # 定义本节可复用的核心函数。
    """COCO crowd criterion: intersection(det,crowd) / area(det)."""  # 执行当前语句以推进本节示例。
    detections, crowds = as_boxes(det_boxes), as_boxes(crowd_boxes)  # 计算并保存当前步骤的中间状态。
    top_left = np.maximum(detections[:, None, :2], crowds[None, :, :2])  # 计算并保存当前步骤的中间状态。
    bottom_right = np.minimum(detections[:, None, 2:], crowds[None, :, 2:])  # 计算并保存当前步骤的中间状态。
    wh = np.maximum(0.0, bottom_right - top_left)  # 计算并保存当前步骤的中间状态。
    intersection = wh[..., 0] * wh[..., 1]  # 计算并保存当前步骤的中间状态。
    det_area = box_area(detections)[:, None]  # 计算并保存当前步骤的中间状态。
    return np.divide(intersection, det_area, out=np.zeros_like(intersection), where=det_area > 0)  # 返回当前分支计算出的结果。

def match_image_class(predictions, ground_truth, iou_threshold):  # 定义本节可复用的核心函数。
    validate_probability_threshold(iou_threshold, "iou_threshold")  # 执行当前语句以推进本节示例。
    if predictions:  # 按当前条件选择后续控制路径。
        validate_detection_arrays([p["box"] for p in predictions],  # 执行当前语句以推进本节示例。
                                  [p["score"] for p in predictions],  # 执行当前语句以推进本节示例。
                                  [0] * len(predictions))  # 执行当前语句以推进本节示例。
    if ground_truth:  # 按当前条件选择后续控制路径。
        as_boxes([g["box"] for g in ground_truth])  # 执行当前语句以推进本节示例。
    predictions = sorted(enumerate(predictions), key=lambda pair: (-pair[1]["score"], pair[0]))  # 计算并保存当前步骤的中间状态。
    regular = [g for g in ground_truth if not g.get("ignore", False) and not g.get("crowd", False)]  # 计算并保存当前步骤的中间状态。
    ignored_regular = [g for g in ground_truth if g.get("ignore", False) and not g.get("crowd", False)]  # 计算并保存当前步骤的中间状态。
    crowds = [g for g in ground_truth if g.get("crowd", False)]  # 计算并保存当前步骤的中间状态。
    used = set()  # 计算并保存当前步骤的中间状态。
    results = []  # 计算并保存当前步骤的中间状态。
    for original_order, prediction in predictions:  # 遍历输入元素以累积或检查结果。
        pbox = [prediction["box"]]  # 计算并保存当前步骤的中间状态。
        regular_iou = box_iou(pbox, [g["box"] for g in regular])[0] if regular else np.array([])  # 计算并保存当前步骤的中间状态。
        candidates = [i for i, value in enumerate(regular_iou)  # 计算并保存当前步骤的中间状态。
                      if value >= iou_threshold and i not in used]  # 按当前条件选择后续控制路径。
        duplicate = any(value >= iou_threshold and i in used for i, value in enumerate(regular_iou))  # 计算并保存当前步骤的中间状态。
        if candidates:  # 按当前条件选择后续控制路径。
            best = max(candidates, key=lambda i: (regular_iou[i], -i))  # 计算并保存当前步骤的中间状态。
            used.add(best)  # 执行当前语句以推进本节示例。
            status, matched, reason = "TP", best, "regular_match"  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            ignored_iou = (box_iou(pbox, [g["box"] for g in ignored_regular])[0]  # 计算并保存当前步骤的中间状态。
                           if ignored_regular else np.array([]))  # 按当前条件选择后续控制路径。
            crowd_iou = (crowd_overlap(pbox, [g["box"] for g in crowds])[0]  # 计算并保存当前步骤的中间状态。
                         if crowds else np.array([]))  # 按当前条件选择后续控制路径。
            if len(ignored_iou) and ignored_iou.max() >= iou_threshold:  # 按当前条件选择后续控制路径。
                status, matched, reason = "IGNORED", None, "ignore_region"  # 计算并保存当前步骤的中间状态。
            elif len(crowd_iou) and crowd_iou.max() >= iou_threshold:  # 按当前条件选择后续控制路径。
                status, matched, reason = "IGNORED", None, "crowd_region"  # 计算并保存当前步骤的中间状态。
            else:  # 处理前置条件不成立的分支。
                status, matched, reason = "FP", None, "duplicate" if duplicate else "unmatched"  # 计算并保存当前步骤的中间状态。
        results.append({"score": float(prediction["score"]), "status": status,  # 执行当前语句以推进本节示例。
                        "reason": reason, "order": original_order,  # 执行当前语句以推进本节示例。
                        "matched_regular": matched})  # 执行当前语句以推进本节示例。
    return results, len(regular)  # 返回当前分支计算出的结果。

img1_cls0_pred = [p for p in PREDICTIONS if p["image_id"] == "img-1" and p["class_id"] == 0]  # 计算并保存当前步骤的中间状态。
img1_cls0_gt = [g for g in GROUND_TRUTH if g["image_id"] == "img-1" and g["class_id"] == 0]  # 计算并保存当前步骤的中间状态。
match_demo, positive_count = match_image_class(img1_cls0_pred, img1_cls0_gt, .5)  # 计算并保存当前步骤的中间状态。
assert [row["status"] for row in match_demo] == ["TP", "FP", "IGNORED"]  # 用受控断言验证关键不变量。
assert match_demo[1]["reason"] == "duplicate"  # 用受控断言验证关键不变量。
assert positive_count == 1  # 用受控断言验证关键不变量。
CONTAINED_CROWD = match_image_class(  # 计算并保存当前步骤的中间状态。
    [{"score": .9, "box": [0, 0, 10, 10]}],  # 执行当前语句以推进本节示例。
    [{"box": [0, 0, 100, 100], "ignore": False, "crowd": True}], .5  # 执行当前语句以推进本节示例。
)[0]  # 执行当前语句以推进本节示例。
assert CONTAINED_CROWD[0]["status"] == "IGNORED"  # 用受控断言验证关键不变量。
print(match_demo)  # 执行当前语句以推进本节示例。


## 7. Precision–Recall 与 AP 协议

按 score 阈值从高到低扫描：`precision=累计TP/(累计TP+累计FP)`，`recall=累计TP/普通GT数`。AP 是 precision–recall 曲线面积，但不同协议并不相同：VOC 2007 曾用 11 点插值，后续 VOC 使用全点积分；COCO 使用 101 个 recall 点，并对多个 IoU、类别、面积和 maxDets 聚合。

因此“mAP=0.42”缺少协议就没有可比性。下面同时实现全点积分与 101 点插值。

In [ ]:
def precision_recall(matched_rows, positives):  # 定义本节可复用的核心函数。
    active = [row for row in matched_rows if row["status"] != "IGNORED"]  # 计算并保存当前步骤的中间状态。
    active = sorted(enumerate(active), key=lambda pair: (-pair[1]["score"], pair[0]))  # 计算并保存当前步骤的中间状态。
    tp = np.array([row["status"] == "TP" for _, row in active], dtype=float)  # 计算并保存当前步骤的中间状态。
    fp = 1.0 - tp  # 计算并保存当前步骤的中间状态。
    cum_tp, cum_fp = np.cumsum(tp), np.cumsum(fp)  # 计算并保存当前步骤的中间状态。
    precision = np.divide(cum_tp, cum_tp + cum_fp, out=np.zeros_like(cum_tp), where=(cum_tp + cum_fp) > 0)  # 计算并保存当前步骤的中间状态。
    recall = cum_tp / positives if positives else np.zeros_like(cum_tp)  # 计算并保存当前步骤的中间状态。
    scores = np.array([row["score"] for _, row in active], dtype=float)  # 计算并保存当前步骤的中间状态。
    return precision, recall, scores  # 返回当前分支计算出的结果。

def interpolated_ap(precision, recall, points=None):  # 定义本节可复用的核心函数。
    precision, recall = np.asarray(precision), np.asarray(recall)  # 计算并保存当前步骤的中间状态。
    if len(precision) == 0:  # 按当前条件选择后续控制路径。
        return 0.0  # 返回当前分支计算出的结果。
    if points is not None:  # 按当前条件选择后续控制路径。
        grid = np.linspace(0, 1, points)  # 计算并保存当前步骤的中间状态。
        values = [precision[recall >= level].max() if np.any(recall >= level) else 0.0 for level in grid]  # 计算并保存当前步骤的中间状态。
        return float(np.mean(values))  # 返回当前分支计算出的结果。
    mrec = np.concatenate([[0.0], recall, [1.0]])  # 计算并保存当前步骤的中间状态。
    mpre = np.concatenate([[0.0], precision, [0.0]])  # 计算并保存当前步骤的中间状态。
    for index in range(len(mpre) - 2, -1, -1):  # 遍历输入元素以累积或检查结果。
        mpre[index] = max(mpre[index], mpre[index + 1])  # 计算并保存当前步骤的中间状态。
    changes = np.flatnonzero(mrec[1:] != mrec[:-1])  # 计算并保存当前步骤的中间状态。
    return float(np.sum((mrec[changes + 1] - mrec[changes]) * mpre[changes + 1]))  # 返回当前分支计算出的结果。

precision_demo, recall_demo, _ = precision_recall(match_demo, positive_count)  # 计算并保存当前步骤的中间状态。
assert np.isclose(interpolated_ap(precision_demo, recall_demo), 1.0)  # 用受控断言验证关键不变量。
assert np.isclose(interpolated_ap(precision_demo, recall_demo, points=101), 1.0)  # 用受控断言验证关键不变量。
print("precision:", precision_demo, "recall:", recall_demo)  # 执行当前语句以推进本节示例。


## 8. 从单图匹配到数据集 mAP

匹配在每个 `image_id × class_id` 内完成，随后该类别的预测跨图片按 score 汇总。分母是该类别所有非 ignore/crowd 真值。最后先对类别求平均，再对 IoU 阈值求平均。

类别没有 GT 时应排除还是计 0，必须由协议规定。本例只评估 fixture 中有普通 GT 的类别。正式评估还需要空预测、空 GT、重复 image id、未知 class id 和最大检测数的契约。

In [ ]:
def evaluate_class(predictions, ground_truth, class_id, iou_threshold):  # 定义本节可复用的核心函数。
    image_ids = sorted({row["image_id"] for row in predictions + ground_truth})  # 计算并保存当前步骤的中间状态。
    matched, positives = [], 0  # 计算并保存当前步骤的中间状态。
    for image_id in image_ids:  # 遍历输入元素以累积或检查结果。
        pred = [p for p in predictions if p["image_id"] == image_id and p["class_id"] == class_id]  # 计算并保存当前步骤的中间状态。
        gt = [g for g in ground_truth if g["image_id"] == image_id and g["class_id"] == class_id]  # 计算并保存当前步骤的中间状态。
        image_rows, image_positives = match_image_class(pred, gt, iou_threshold)  # 计算并保存当前步骤的中间状态。
        matched.extend(image_rows)  # 执行当前语句以推进本节示例。
        positives += image_positives  # 计算并保存当前步骤的中间状态。
    precision, recall, scores = precision_recall(matched, positives)  # 计算并保存当前步骤的中间状态。
    return {"ap_all_points": interpolated_ap(precision, recall),  # 返回当前分支计算出的结果。
            "ap_101": interpolated_ap(precision, recall, 101),  # 执行当前语句以推进本节示例。
            "precision": precision, "recall": recall, "scores": scores,  # 执行当前语句以推进本节示例。
            "positives": positives, "matched": matched}  # 执行当前语句以推进本节示例。

def evaluate_map(predictions, ground_truth, classes=(0, 1), thresholds=(.5, .75)):  # 定义本节可复用的核心函数。
    details = {}  # 计算并保存当前步骤的中间状态。
    for threshold in thresholds:  # 遍历输入元素以累积或检查结果。
        for class_id in classes:  # 遍历输入元素以累积或检查结果。
            details[(threshold, class_id)] = evaluate_class(  # 计算并保存当前步骤的中间状态。
                predictions, ground_truth, class_id, threshold  # 执行当前语句以推进本节示例。
            )  # 执行当前语句以推进本节示例。
    mean_ap = float(np.mean([row["ap_101"] for row in details.values()]))  # 计算并保存当前步骤的中间状态。
    return mean_ap, details  # 返回当前分支计算出的结果。

mean_ap, map_details = evaluate_map(PREDICTIONS, GROUND_TRUTH)  # 计算并保存当前步骤的中间状态。
print("teaching mAP@[.50,.75]:", round(mean_ap, 4),  # 执行当前语句以推进本节示例。
      {str(key): round(value["ap_101"], 3) for key, value in map_details.items()})  # 执行当前语句以推进本节示例。
assert 0.0 <= mean_ap <= 1.0  # 用受控断言验证关键不变量。
assert map_details[(.5, 0)]["positives"] == 2  # 用受控断言验证关键不变量。
assert map_details[(.5, 1)]["positives"] == 1  # 用受控断言验证关键不变量。


## 9. 置信阈值、NMS 阈值与延迟要一起看

降低 score 阈值提高候选召回，也增加 NMS 与下游存储成本；降低 NMS IoU 阈值减少重复框，也可能压掉密集目标。AP 通常需要保留较低分候选形成完整 PR 曲线，不能先用线上高阈值截断后再声称官方 AP。

下面的 sweep 只是证明参数如何影响候选数与教学 mAP。微秒级计时受机器噪声影响，只能验证计时代码路径；生产需在真实 batch、输入尺寸、硬件和并发下报告 p50/p95/p99。

In [ ]:
def postprocess_fixture(predictions, score_threshold, nms_iou):  # 定义本节可复用的核心函数。
    output = []  # 计算并保存当前步骤的中间状态。
    for image_id in sorted({p["image_id"] for p in predictions}):  # 遍历输入元素以累积或检查结果。
        rows = [p for p in predictions if p["image_id"] == image_id and p["score"] >= score_threshold]  # 计算并保存当前步骤的中间状态。
        if not rows:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        keep = greedy_class_aware_nms(  # 计算并保存当前步骤的中间状态。
            [r["box"] for r in rows], [r["score"] for r in rows],  # 执行当前语句以推进本节示例。
            [r["class_id"] for r in rows], nms_iou  # 执行当前语句以推进本节示例。
        )  # 执行当前语句以推进本节示例。
        output.extend(rows[i] for i in keep)  # 执行当前语句以推进本节示例。
    return output  # 返回当前分支计算出的结果。

sweep = []  # 计算并保存当前步骤的中间状态。
for score_threshold in (.05, .50, .75):  # 遍历输入元素以累积或检查结果。
    for nms_iou in (.3, .5, .7):  # 遍历输入元素以累积或检查结果。
        start = time.perf_counter()  # 计算并保存当前步骤的中间状态。
        post = postprocess_fixture(PREDICTIONS, score_threshold, nms_iou)  # 计算并保存当前步骤的中间状态。
        elapsed_us = (time.perf_counter() - start) * 1e6  # 计算并保存当前步骤的中间状态。
        score, _ = evaluate_map(post, GROUND_TRUTH)  # 计算并保存当前步骤的中间状态。
        sweep.append((score_threshold, nms_iou, len(post), score, elapsed_us))  # 执行当前语句以推进本节示例。
print("score_thr nms_iou count teaching_mAP elapsed_us")  # 执行当前语句以推进本节示例。
for row in sweep:  # 遍历输入元素以累积或检查结果。
    print(tuple(round(value, 4) if isinstance(value, float) else value for value in row))  # 执行当前语句以推进本节示例。
assert all(row[2] <= len(PREDICTIONS) for row in sweep)  # 用受控断言验证关键不变量。


## 10. AP 下降要拆成可行动错误

常见类别：`background`（不覆盖任何 GT）、`localization`（类别对但 IoU 不足）、`classification`（位置对但类别错）、`duplicate`（普通 GT 已被更高分预测占用）、`miss`（没有任何预测覆盖 GT）。

诊断时比较多个 IoU：在 0.5 是 TP、0.75 变 FP，通常是定位；在所有阈值都失败可能是分类或召回。还要按目标面积、遮挡、场景、设备和时间分群，而非只看一张 PR 曲线。

In [ ]:
def detection_error_analysis(predictions, ground_truth, iou_threshold=.5):  # 定义本节可复用的核心函数。
    """先复用 matching，再只对 unmatched FP 做跨类/定位诊断。"""  # 执行当前语句以推进本节示例。
    validate_probability_threshold(iou_threshold, "iou_threshold")  # 执行当前语句以推进本节示例。
    records, matched_gt = [], set()  # 计算并保存当前步骤的中间状态。
    groups = sorted({(r["image_id"], r["class_id"]) for r in predictions + ground_truth})  # 计算并保存当前步骤的中间状态。
    for image_id, class_id in groups:  # 遍历输入元素以累积或检查结果。
        indexed = [(i, p) for i, p in enumerate(predictions)  # 计算并保存当前步骤的中间状态。
                   if p["image_id"] == image_id and p["class_id"] == class_id]  # 按当前条件选择后续控制路径。
        class_gt = [g for g in ground_truth  # 计算并保存当前步骤的中间状态。
                    if g["image_id"] == image_id and g["class_id"] == class_id]  # 按当前条件选择后续控制路径。
        regular = [g for g in class_gt if not g.get("ignore", False) and not g.get("crowd", False)]  # 计算并保存当前步骤的中间状态。
        matched_rows, _ = match_image_class([p for _, p in indexed], class_gt, iou_threshold)  # 计算并保存当前步骤的中间状态。
        for row in matched_rows:  # 遍历输入元素以累积或检查结果。
            global_index, prediction = indexed[row["order"]]  # 计算并保存当前步骤的中间状态。
            if row["status"] == "TP":  # 按当前条件选择后续控制路径。
                matched_gt.add((image_id, class_id, row["matched_regular"]))  # 执行当前语句以推进本节示例。
                kind = "tp"  # 计算并保存当前步骤的中间状态。
            elif row["status"] == "IGNORED":  # 按当前条件选择后续控制路径。
                kind = "ignored"  # 计算并保存当前步骤的中间状态。
            elif row["reason"] == "duplicate":  # 按当前条件选择后续控制路径。
                kind = "duplicate"  # 计算并保存当前步骤的中间状态。
            else:  # 处理前置条件不成立的分支。
                candidates = [g for g in ground_truth if g["image_id"] == image_id  # 计算并保存当前步骤的中间状态。
                              and not g.get("ignore", False) and not g.get("crowd", False)]  # 执行当前语句以推进本节示例。
                if not candidates:  # 按当前条件选择后续控制路径。
                    kind = "background"  # 计算并保存当前步骤的中间状态。
                else:  # 处理前置条件不成立的分支。
                    overlaps = box_iou([prediction["box"]], [g["box"] for g in candidates])[0]  # 计算并保存当前步骤的中间状态。
                    best = int(np.argmax(overlaps))  # 计算并保存当前步骤的中间状态。
                    target = candidates[best]  # 计算并保存当前步骤的中间状态。
                    if overlaps[best] < .1:  # 按当前条件选择后续控制路径。
                        kind = "background"  # 计算并保存当前步骤的中间状态。
                    elif target["class_id"] != class_id and overlaps[best] >= iou_threshold:  # 按当前条件选择后续控制路径。
                        kind = "classification"  # 计算并保存当前步骤的中间状态。
                    elif target["class_id"] == class_id and overlaps[best] < iou_threshold:  # 按当前条件选择后续控制路径。
                        kind = "localization"  # 计算并保存当前步骤的中间状态。
                    else:  # 处理前置条件不成立的分支。
                        kind = "background"  # 计算并保存当前步骤的中间状态。
            records.append({"type": kind, "prediction_index": global_index,  # 执行当前语句以推进本节示例。
                            "image_id": image_id, "class_id": class_id,  # 执行当前语句以推进本节示例。
                            "match_status": row["status"], "match_reason": row["reason"]})  # 执行当前语句以推进本节示例。
        for local_index, gt in enumerate(regular):  # 遍历输入元素以累积或检查结果。
            if (image_id, class_id, local_index) not in matched_gt:  # 按当前条件选择后续控制路径。
                records.append({"type": "miss", "prediction_index": None,  # 执行当前语句以推进本节示例。
                                "image_id": image_id, "class_id": class_id, "gt_box": gt["box"]})  # 执行当前语句以推进本节示例。
    return records  # 返回当前分支计算出的结果。

ERROR_RECORDS = detection_error_analysis(PREDICTIONS, GROUND_TRUTH)  # 计算并保存当前步骤的中间状态。
error_taxonomy = Counter(row["type"] for row in ERROR_RECORDS)  # 计算并保存当前步骤的中间状态。
CLASSIFICATION_PROBE = detection_error_analysis(  # 计算并保存当前步骤的中间状态。
    [{"image_id": "img-1", "class_id": 1, "score": .2, "box": [10, 10, 50, 50]}],  # 执行当前语句以推进本节示例。
    GROUND_TRUTH, .5  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
print(error_taxonomy)  # 执行当前语句以推进本节示例。
assert error_taxonomy["duplicate"] >= 1  # 用受控断言验证关键不变量。
assert error_taxonomy["ignored"] >= 1  # 用受控断言验证关键不变量。
assert error_taxonomy["background"] >= 1  # 用受控断言验证关键不变量。
assert any(row["type"] == "classification" for row in CLASSIFICATION_PROBE)  # 用受控断言验证关键不变量。
assert all(not (row["type"] in {"classification", "localization"} and row["match_status"] == "IGNORED")  # 用受控断言验证关键不变量。
           for row in ERROR_RECORDS if "match_status" in row)  # 遍历输入元素以累积或检查结果。


## 11. 把后处理配置变成版本化组件

模型导出的 raw boxes 不是稳定业务结果。score threshold、坐标解码、clip、最小面积、NMS 类型和阈值都会改变输出，应与模型形成不可分割的 bundle。返回值还应包含原图尺寸和变换 trace，避免调用方二次误缩放。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class DetectionPostprocessor:  # 定义承载本节状态与行为的数据结构。
    score_threshold: float = .25  # 计算并保存当前步骤的中间状态。
    nms_iou: float = .50  # 计算并保存当前步骤的中间状态。
    min_area: float = 1.0  # 计算并保存当前步骤的中间状态。
    max_detections: int = 100  # 计算并保存当前步骤的中间状态。
    version: str = "xyxy-halfopen-classnms-v2"  # 计算并保存当前步骤的中间状态。

    def __post_init__(self):  # 定义本节可复用的核心函数。
        validate_probability_threshold(self.score_threshold, "score_threshold")  # 执行当前语句以推进本节示例。
        validate_probability_threshold(self.nms_iou, "nms_iou")  # 执行当前语句以推进本节示例。
        if not np.isfinite(self.min_area) or self.min_area < 0:  # 按当前条件选择后续控制路径。
            raise ValueError("min_area 必须是有限非负数")  # 遇到非法合同立即显式失败。
        if isinstance(self.max_detections, bool) or not isinstance(self.max_detections, (int, np.integer)) or self.max_detections <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("max_detections 必须是正整数")  # 遇到非法合同立即显式失败。
        if not self.version:  # 按当前条件选择后续控制路径。
            raise ValueError("version 不能为空")  # 遇到非法合同立即显式失败。

    def __call__(self, boxes, scores, class_ids, image_hw):  # 定义本节可复用的核心函数。
        validate_image_hw(image_hw)  # 执行当前语句以推进本节示例。
        boxes, scores, class_ids = validate_detection_arrays(boxes, scores, class_ids)  # 计算并保存当前步骤的中间状态。
        boxes = clip_boxes(boxes, image_hw)  # 计算并保存当前步骤的中间状态。
        valid = (box_area(boxes) >= self.min_area) & (scores >= self.score_threshold)  # 计算并保存当前步骤的中间状态。
        original = np.flatnonzero(valid)  # 计算并保存当前步骤的中间状态。
        if not len(original):  # 按当前条件选择后续控制路径。
            return []  # 返回当前分支计算出的结果。
        relative = greedy_class_aware_nms(boxes[valid], scores[valid], class_ids[valid], self.nms_iou)  # 计算并保存当前步骤的中间状态。
        keep = original[relative][:self.max_detections]  # 计算并保存当前步骤的中间状态。
        return [{"box": boxes[i].tolist(), "score": float(scores[i]),  # 返回当前分支计算出的结果。
                 "class_id": int(class_ids[i]), "postprocess_version": self.version}  # 执行当前语句以推进本节示例。
                for i in keep]  # 遍历输入元素以累积或检查结果。

postprocessor = DetectionPostprocessor()  # 计算并保存当前步骤的中间状态。
served = postprocessor(probe_boxes, probe_scores, probe_classes, (10, 10))  # 计算并保存当前步骤的中间状态。
assert len(served) == 2  # 用受控断言验证关键不变量。
assert all(row["postprocess_version"] == postprocessor.version for row in served)  # 用受控断言验证关键不变量。
print(served)  # 执行当前语句以推进本节示例。


## 12. 边界与回归测试

检测测试不能只放一组正常框。最低集合包括：完全重合、不相交、仅接边、包含关系、退化框、负坐标、越界、resize 往返、同分排序、跨类别重合、重复检测、ignore/crowd、多图片全局排序和空输出。

In [ ]:
assert np.allclose(xywh_to_xyxy([[2, 3, 5, 7]]), [[2, 3, 7, 10]])  # 用受控断言验证关键不变量。
assert np.allclose(xyxy_to_xywh([[2, 3, 7, 10]]), [[2, 3, 5, 7]])  # 用受控断言验证关键不变量。
assert np.allclose(clip_boxes([[-2, -3, 12, 11]], (10, 10)), [[0, 0, 10, 10]])  # 用受控断言验证关键不变量。
assert np.allclose(letterbox_inverse(*letterbox_map([[3, 4, 30, 40]], (50, 80), (640, 640))), [[3, 4, 30, 40]])  # 用受控断言验证关键不变量。
assert box_area([[0, 0, 0, 5]])[0] == 0  # 用受控断言验证关键不变量。
assert np.isclose(box_iou([[0, 0, 10, 10]], [[2, 2, 8, 8]])[0, 0], .36)  # 用受控断言验证关键不变量。
assert np.allclose(box_iou([[0, 0, 5, 5]], [[0, 0, 5, 5], [5, 0, 10, 5]]), [[1, 0]])  # 用受控断言验证关键不变量。
assert greedy_class_aware_nms(probe_boxes, probe_scores, probe_classes, .5).tolist() == [0, 2]  # 用受控断言验证关键不变量。
tie_keep = greedy_class_aware_nms([[0, 0, 10, 10], [1, 1, 9, 9]], [.8, .8], [0, 0], .5)  # 计算并保存当前步骤的中间状态。
assert tie_keep.tolist() == [0]  # 用受控断言验证关键不变量。
assert soft_scores[1] < .8 and soft_scores[1] >= 0  # 用受控断言验证关键不变量。
assert [row["status"] for row in match_demo].count("TP") == 1  # 用受控断言验证关键不变量。
assert [row["reason"] for row in match_demo].count("duplicate") == 1  # 用受控断言验证关键不变量。
crowd_pred = [p for p in PREDICTIONS if p["image_id"] == "img-2" and p["class_id"] == 1]  # 计算并保存当前步骤的中间状态。
crowd_gt = [g for g in GROUND_TRUTH if g["image_id"] == "img-2" and g["class_id"] == 1]  # 计算并保存当前步骤的中间状态。
crowd_match, crowd_positive = match_image_class(crowd_pred, crowd_gt, .5)  # 计算并保存当前步骤的中间状态。
assert crowd_positive == 0  # 用受控断言验证关键不变量。
assert all(row["status"] == "IGNORED" and row["reason"] == "crowd_region" for row in crowd_match)  # 用受控断言验证关键不变量。
assert crowd_overlap([[0, 0, 10, 10]], [[0, 0, 100, 100]])[0, 0] == 1.0  # 用受控断言验证关键不变量。
assert CONTAINED_CROWD[0]["status"] == "IGNORED"  # 用受控断言验证关键不变量。
assert np.all(np.diff(map_details[(.5, 0)]["recall"]) >= 0)  # 用受控断言验证关键不变量。
assert 0 <= interpolated_ap([1, .5], [.5, 1.0]) <= 1  # 用受控断言验证关键不变量。
assert len(postprocessor([], [], [], (10, 10))) == 0  # 用受控断言验证关键不变量。
assert error_taxonomy["duplicate"] >= 1 and error_taxonomy["ignored"] >= 1  # 用受控断言验证关键不变量。
assert any(row["type"] == "classification" for row in CLASSIFICATION_PROBE)  # 用受控断言验证关键不变量。

for bad_call in [  # 遍历输入元素以累积或检查结果。
    lambda: as_boxes([[5, 5, 4, 8]]),  # 执行当前语句以推进本节示例。
    lambda: xywh_to_xyxy([[np.nan, 0, 1, 1]]),  # 执行当前语句以推进本节示例。
    lambda: clip_boxes([[0, 0, 1, 1]], (0, 10)),  # 执行当前语句以推进本节示例。
    lambda: letterbox_map([[0, 0, 1, 1]], (10, 0), (10, 10)),  # 执行当前语句以推进本节示例。
    lambda: greedy_class_aware_nms(probe_boxes, probe_scores[:2], probe_classes, .5),  # 执行当前语句以推进本节示例。
    lambda: greedy_class_aware_nms(probe_boxes, [np.nan, .8, .7], probe_classes, .5),  # 执行当前语句以推进本节示例。
    lambda: linear_soft_nms(probe_boxes, probe_scores, probe_classes[:2]),  # 执行当前语句以推进本节示例。
    lambda: linear_soft_nms(probe_boxes, probe_scores, probe_classes, iou_threshold=1.5),  # 计算并保存当前步骤的中间状态。
    lambda: DetectionPostprocessor(nms_iou=1.5),  # 计算并保存当前步骤的中间状态。
    lambda: DetectionPostprocessor(min_area=-1),  # 计算并保存当前步骤的中间状态。
    lambda: DetectionPostprocessor(max_detections=0),  # 计算并保存当前步骤的中间状态。
]:  # 执行当前语句以推进本节示例。
    try:  # 尝试执行可能失败的受控操作。
        bad_call()  # 执行当前语句以推进本节示例。
        raise AssertionError("非法检测输入应被拒绝")  # 遇到非法合同立即显式失败。
    except ValueError:  # 捕获预期异常并验证失败分支。
        pass  # 调整当前循环或占位控制流。
print("目标检测契约测试通过：几何、crowd、匹配、指标与非法输入均已覆盖")  # 执行当前语句以推进本节示例。


## 13. 生产边界与原始资料

**不要混淆的边界**：本例是轴对齐框；不是 mask/旋转框。crowd 逻辑是教学简化；不是 COCO evaluator 等价实现。小型 fixture 能证明数学与契约；不能预测真实模型质量。微基准能检查路径；不能代表服务尾延迟。

**上线清单**：固定坐标与 resize 版本；黄金图往返误差；原始输出和最终输出可追踪；score/NMS/最大检测数联合回归；官方 evaluator 容器与版本锁定；按类别/面积/场景误差分析；空图、超大图、NaN、未知类保护；真实并发 p50/p95/p99；灰度与回滚。

**资料**：

- Everingham et al., *The PASCAL Visual Object Classes Challenge*, IJCV 2010：https://doi.org/10.1007/s11263-009-0275-4
- Lin et al., *Microsoft COCO: Common Objects in Context*, ECCV 2014：https://arxiv.org/abs/1405.0312
- COCO 官方 API / evaluator：https://github.com/cocodataset/cocoapi
- Neubeck & Van Gool, *Efficient Non-Maximum Suppression*, ICPR 2006：https://doi.org/10.1109/ICPR.2006.479
- Bodla et al., *Soft-NMS — Improving Object Detection With One Line of Code*, ICCV 2017：https://arxiv.org/abs/1704.04503